# Examen Final

## Leandro Bravo  GR1CC

### Diseño e Implementación de un Sistema de Recuperación de Información

## Importar Librerías

In [46]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
import time
import os
import google.generativeai as genai
from dotenv import load_dotenv
import gradio as gr

## Carga del Dataset

In [4]:
# Download latest version
path = kagglehub.dataset_download("spsayakpaul/arxiv-paper-abstracts")

print("Path to dataset files:", path)

100%|██████████| 44.6M/44.6M [01:39<00:00, 471kB/s] 

Extracting files...


Path to dataset files: C:\Users\leand\.cache\kagglehub\datasets\spsayakpaul\arxiv-paper-abstracts\versions\2


## A. Preparación del corpus

In [15]:
# Asumiendo que el archivo se llama arxiv_data.csv o arxiv-metadata-oai-snapshot.json
df = pd.read_csv(path + '/arxiv_data.csv') 

df = df.sample(10000, random_state=42).reset_index(drop=True)

# 2. Limpiar valores nulos y crear el texto a procesar
df = df.fillna('')

df['text_to_embed'] = df['titles'] + ". " + df['summaries']

# Lista de textos que usaremos
corpus = df['text_to_embed'].tolist()
print(f"Total de documentos a procesar: {len(corpus)}")


Total de documentos a procesar: 10000


## B. Representación mediante embeddings

In [16]:
print("Cargando modelo de embeddings (Bi-Encoder)...")
# 'all-MiniLM-L6-v2' es ligero, rápido y excelente para procesar abstracts científicos
retriever_model = SentenceTransformer('all-MiniLM-L6-v2')

print("Generando embeddings del corpus (esto puede tardar un par de minutos)...")
# Generamos los embeddings y los convertimos a numpy array, formato requerido por FAISS
corpus_embeddings = retriever_model.encode(corpus, show_progress_bar=True, convert_to_numpy=True)

Cargando modelo de embeddings (Bi-Encoder)...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5424.85it/s]


Generando embeddings del corpus (esto puede tardar un par de minutos)...


Batches: 100%|██████████| 313/313 [03:54<00:00,  1.33it/s]


## C. Almacenamiento y búsqueda vectorial

In [17]:
# 1. Normalizamos los vectores para poder usar Distancia de Producto Interno como Similitud del Coseno
faiss.normalize_L2(corpus_embeddings)

# 2. Extraemos la dimensión de los vectores (para all-MiniLM es 384)
dimension = corpus_embeddings.shape[1]

# 3. Construimos el índice FAISS (IP = Inner Product)
index = faiss.IndexFlatIP(dimension)
index.add(corpus_embeddings)

print(f"Índice FAISS de Búsqueda Vectorial creado exitosamente con {index.ntotal} documentos.")

Índice FAISS de Búsqueda Vectorial creado exitosamente con 10000 documentos.


## D. Recuperación

### D.1: Carga del Modelo de Re-ranking (Cross-Encoder)

In [54]:
print("Cargando modelo de Re-ranking (Cross-Encoder)")
# Usamos un modelo entrenado en el dataset MS MARCO (diseñado para búsqueda y recuperación)
reranker_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
print("Modelo de Re-ranking cargado y listo.")

Cargando modelo de Re-ranking (Cross-Encoder)


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 11396.98it/s]


Modelo de Re-ranking cargado y listo.


### D.2: Función de Recuperación Avanzada

In [55]:
def recuperar_documentos(query, top_k_inicial=50, top_k_final=3):
    start_time = time.time()
    
    # FASE 1: BÚSQUEDA VECTORIAL RÁPIDA (BI-ENCODER + FAISS)
    # Convertimos la consulta a vector y la normalizamos
    query_embedding = retriever_model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(query_embedding)
    
    # Buscamos en el índice FAISS (devuelve distancias e índices de los documentos)
    distancias, indices = index.search(query_embedding, top_k_inicial)
    
    candidatos_indices = indices[0]
    candidatos_textos = [corpus[i] for i in candidatos_indices]
    
    # FASE 2: RE-RANKING SEMÁNTICO (CROSS-ENCODER)
    # Preparamos los pares [Consulta, Documento] para el Cross-Encoder
    pares_evaluacion = [[query, texto] for texto in candidatos_textos]
    
    # El modelo predice un puntaje de relevancia para cada par
    scores_rerank = reranker_model.predict(pares_evaluacion)
    
    # Ordenamos de mayor a menor puntaje
    resultados_ordenados = sorted(
        zip(candidatos_indices, scores_rerank), 
        key=lambda x: x[1], 
        reverse=True
    )
    
    # FILTRADO Y EXTRACCIÓN FINALES
    # Tomamos solo el top_k_final
    top_resultados = resultados_ordenados[:top_k_final]
    
    documentos_recuperados = []
    
    for idx, score in top_resultados:
        # Extraemos la data original del DataFrame usando el índice
        # Modificado para usar los nombres de columnas de tu dataset (titles y summaries)
        titulo = df.iloc[idx]['titles']
        abstract = df.iloc[idx]['summaries']
        
        documentos_recuperados.append({
            "titulo": titulo,
            "abstract": abstract,
            "score": float(score)
        })
        
    tiempo_total = time.time() - start_time
    # Imprimimos logs útiles para probar la función
    print(f"Recuperación completada en {tiempo_total:.2f} segundos para la consulta: '{query}'")
    
    return documentos_recuperados

In [56]:
query_prueba = "What are the main applications of Graph Neural Networks?"
resultados = recuperar_documentos(query_prueba)

for i, doc in enumerate(resultados):
    print(f"\n[{i+1}] Score: {doc['score']:.2f} | Título: {doc['titulo']}")
    print(f"Resumen: {doc['abstract'][:150]}... +")

Recuperación completada en 1.42 segundos para la consulta: 'What are the main applications of Graph Neural Networks?'

[1] Score: 4.94 | Título: AutoGraph: Automated Graph Neural Network
Resumen: Graphs play an important role in many applications. Recently, Graph Neural
Networks (GNNs) have achieved promising results in graph analysis tasks. So... +

[2] Score: 4.88 | Título: The Logic of Graph Neural Networks
Resumen: Graph neural networks (GNNs) are deep learning architectures for machine
learning problems on graphs. It has recently been shown that the expressivene... +

[3] Score: 4.62 | Título: A Practical Guide to Graph Neural Networks
Resumen: Graph neural networks (GNNs) have recently grown in popularity in the field
of artificial intelligence due to their unique ability to ingest relativel... +


## E. Generación aumentada por recuperación

### E.1: Configuración de Gemini

In [57]:
load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    print("⚠️ ADVERTENCIA: No se encontró la clave de API. Asegúrate de crear el archivo .env")
else:
    genai.configure(api_key=GEMINI_API_KEY)
    print("API de Gemini configurada exitosamente.")

# Definir el System Prompt estricto
instrucciones_sistema = """Eres un asistente académico experto. Tu tarea es responder la pregunta del usuario utilizando ÚNICAMENTE la información proporcionada en la sección de 'Contextos'.
    
REGLAS ESTRICTAS:
1. Si la información en los contextos no es suficiente para responder la pregunta, o no hay contextos relevantes, debes responder EXACTAMENTE con esta frase: "El corpus no contiene información suficiente para responder a esta consulta."
2. No utilices conocimiento externo ni alucines información.
3. Cuando uses información de un contexto, SIEMPRE cita la fuente al final de la oración indicando el número de documento, por ejemplo: [Documento 1] o [Documento 1, Documento 2].
"""

# Inicializar el modelo
modelo_generativo = genai.GenerativeModel(
    model_name="gemini-1.5-flash-latest",
    system_instruction=instrucciones_sistema
)

API de Gemini configurada exitosamente.


### E.2: Función de Generación con LLM

In [58]:
def generar_respuesta_rag(query, documentos_recuperados):
    if not documentos_recuperados:
         return "El corpus no contiene información suficiente para responder a esta consulta."
         
    # 1. Construir el bloque de contexto estructurado
    texto_contextos = ""
    for i, doc in enumerate(documentos_recuperados):
        texto_contextos += f"Documento {i+1}\n"
        texto_contextos += f"Título: {doc['titulo']}\n"
        texto_contextos += f"Resumen: {doc['abstract']}\n\n"
        
    # 2. Armar el prompt final para el usuario
    prompt_usuario = f"Contextos:\n{texto_contextos}\n\nPregunta: {query}"
    
    # 3. Configurar la generación (Temperatura muy baja para asegurar fidelidad)
    configuracion = genai.GenerationConfig(
        temperature=0.1
    )
    
    # 4. Solicitar la respuesta a Gemini
    try:
        respuesta = modelo_generativo.generate_content(
            prompt_usuario,
            generation_config=configuracion
        )
        return respuesta.text
    except Exception as e:
        return f"Error en la generación de respuesta: {str(e)}"

### E.3 y F: Integración y Presentación de Evidencias

In [59]:
def pipeline_completo_rag(query):
    print(f"Procesando consulta: '{query}'...\n")
    
    # 1. Fase de Recuperació
    documentos = recuperar_documentos(query, top_k_inicial=50, top_k_final=3)
    
    # 2. Fase de Generación 
    respuesta_llm = generar_respuesta_rag(query, documentos)
    
    # 3. Mostrar Resultados y Evidencias (Requerimiento F)
    print("RESPUESTA GENERADA:\n")
    print(respuesta_llm +"\n")
    print("EVIDENCIAS UTILIZADAS:\n")
    for i, doc in enumerate(documentos):
        print(f"[{i+1}] Score: {doc['score']:.4f} | Título: {doc['titulo']}")
        # Imprimimos los primeros 250 caracteres del abstract para verificar visualmente
        print(f"    Resumen: {doc['abstract'][:250]}...\n")
    print("="*60)
    
    return respuesta_llm, documentos

In [60]:
query_prueba = "How is reinforcement learning used in robotics?"
respuesta, evidencias = pipeline_completo_rag(query_prueba)

Procesando consulta: 'How is reinforcement learning used in robotics?'...

Recuperación completada en 1.39 segundos para la consulta: 'How is reinforcement learning used in robotics?'
RESPUESTA GENERADA:

Error en la generación de respuesta: 404 models/gemini-1.5-flash-latest is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.

EVIDENCIAS UTILIZADAS:

[1] Score: 7.3418 | Título: Using Deep Reinforcement Learning for the Continuous Control of Robotic Arms
    Resumen: Deep reinforcement learning enables algorithms to learn complex behavior,
deal with continuous action spaces and find good strategies in environments
with high dimensional state spaces. With deep reinforcement learning being an
active area of researc...

[2] Score: 6.4164 | Título: Autonomous Reinforcement Learning of Multiple Interrelated Tasks
    Resumen: Autonomous multiple tasks learning is a fundame

## G. Interfaz Gráfica

In [61]:
def funcion_interfaz_rag(mensaje, historial):

    # 1. Ejecutar la recuperación
    documentos_recuperados = recuperar_documentos(mensaje, top_k_inicial=50, top_k_final=3)
    
    # 2. Generar la respuesta usando Gemini
    respuesta_llm = generar_respuesta_rag(mensaje, documentos_recuperados)
    
    # 3. Formatear la salida final combinando Respuesta + Evidencias (Requerimiento G)
    origen_evidencias = ""
    if documentos_recuperados and "El corpus no contiene información suficiente" not in respuesta_llm:
        origen_evidencias += "\n\n---\n### Evidencias utilizadas para construir la respuesta:\n"
        for i, doc in enumerate(documentos_recuperados):
            origen_evidencias += f"**[{i+1}] {doc['titulo']}** *(Score de relevancia: {doc['score']:.2f})*\n"
            origen_evidencias += f"> {doc['abstract']}\n\n"
    else:
        origen_evidencias += "\n\n---\n### Evidencias:\n*Ninguna evidencia del corpus fue suficientemente relevante para esta consulta.*"
    
    # Unimos la respuesta del modelo con sus respectivas evidencias en la interfaz gráfica
    return f"{respuesta_llm}{origen_evidencias}"

# 4. Configurar el diseño de la interfaz tipo Chat
demo = gr.ChatInterface(
    fn=funcion_interfaz_rag,
    title=" Sistema RAG de Recuperación de Información - arXiv",
    description="Asistente conversacional para consultas sobre resúmenes de artículos científicos de arXiv utilizando búsquedas vectoriales y Gemini.",
    textbox=gr.Textbox(placeholder="Escribe tu consulta aquí en Inglés... (ej. What are the main applications of Graph Neural Networks?)", container=False, scale=7),
    examples=[
        "What are the main applications of Graph Neural Networks?",
        "How is reinforcement learning used in robotics?",
        "Recent advances in diffusion models for image generation.",
        "Techniques for improving retrieval-augmented generation systems.",
        "How to bake a chocolate cake?" # Prueba de control para verificar que responda que no hay suficiente información
    ],
    submit_btn="Enviar",
    stop_btn="Detener",
)

# 5. Lanzar la aplicación de manera local dentro del cuaderno
# Usamos inline=True para visualizar el chat directamente en la celda de VS Code
demo.launch(inline=True, share=False)

C:\Users\leand\AppData\Local\Temp\ipykernel_23584\3091813354.py:23: UserWarning: You provided a custom `textbox` component, but also specified `submit_btn`, `stop_btn` parameter(s) on `gr.ChatInterface`. These ChatInterface parameters will be ignored. To customize these settings, pass them directly to your `gr.Textbox` or `gr.MultimodalTextbox` component instead. For example: textbox=gr.Textbox(..., submit_btn='submit')
  demo = gr.ChatInterface(


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


## H. Despliegue en la nube

### Paso 1: Exportar la muestra de datos

In [62]:
# Guardamos solo las columnas necesarias en un archivo ligero
df_export = df[['titles', 'summaries', 'text_to_embed']]
df_export.to_csv('arxiv_sample.csv', index=False)
print("¡Archivo arxiv_sample.csv generado con éxito!")

¡Archivo arxiv_sample.csv generado con éxito!


### Paso 2: Crear los archivos de despliegue

Se creó los archivos arxiv_sample.csv y el requirements

### Paso 3: Configurar Render.com